# 🔺 Delta Lake — Advanced Optimization: Partitioning

---

## 1. Data Cardinality vs Estratégia de Otimização
![Foto](./Fotos/11.png)
![Foto](./Fotos/12.png)
![Foto](./Fotos/13.png)
![Foto](./Fotos/14.png)
**Cardinalidade** é o número de valores únicos em uma coluna.  
Ela é o principal fator na escolha da estratégia de otimização correta.

```
  Baixa cardinalidade           Alta cardinalidade
  ──────────────────            ─────────────────────
  ex: Year, Month, Category     ex: UserId, OrderId, Timestamp
  poucos valores únicos         muitos valores únicos
  boa candidata a partição      ruim para partição → Z-Ordering
```

| Cardinalidade da coluna | Estratégia recomendada |
|---|---|
| Baixa / Média (ex: Year, Country) | **Partitioning** — cria subdiretórios físicos por valor |
| Alta (ex: UserId, ProductId) | **Z-Ordering** — co-localiza dados relacionados dentro de arquivos |
| Qualquer | **Data Skipping via `_delta_log`** — automático, sempre ativo |

---

## 2. Partitioning — Conceitos Fundamentais

O **particionamento** divide os dados em subdiretórios físicos com base nos valores de uma coluna.  
Ao filtrar por essa coluna, o Spark lê apenas o subdiretório correspondente — ignorando todos os outros.

```
  Tabela SEM partição:           Tabela COM partição por Year:

  flight_data/                   flight_data_year/
  ├── _delta_log/                ├── _delta_log/          ← único, na raiz
  ├── part-001.parquet           ├── Year=2007/
  ├── part-002.parquet           │     └── part-001.parquet
  └── part-003.parquet           └── Year=2008/
                                        └── part-002.parquet

  WHERE year = 2007 →            WHERE year = 2007 →
  _delta_log stats eliminam      Spark lê APENAS Year=2007/
  arquivos desnecessários        sem nem consultar Year=2008/
```

---

### Quando (e quando não) particionar

A Databricks tem recomendações claras sobre quando o particionamento vale a pena:

```
  ┌─────────────────────────────────────────────────────────────┐
  │  Tabela < 1 TB?                                             │
  │       │                                                     │
  │      SIM ──► NÃO particione                                 │
  │       │      O Delta Log + File Skipping já são suficientes │
  │       │      Particionamento pode até piorar a performance  │
  │       │                                                     │
  │      NÃO ──► Cada partição terá pelo menos 1 GB de dados?  │
  │                   │                                         │
  │                  SIM ──► Escolha a coluna com cuidado       │
  │                   │                                         │
  │                  NÃO ──► NÃO particione                     │
  │                           Risco de small files por partição │
  └─────────────────────────────────────────────────────────────┘
```

---

### Escolhendo a coluna de partição — o problema de distribuição

Baixa cardinalidade é necessária, mas **não suficiente**.  
A distribuição dos dados entre os valores da coluna é igualmente importante.

```
  Coluna: Category

  Partição Electronics: ████████████████████████████████ 99% dos dados
  Partição Books:       ▌                                 0.5%
  Partição Sports:      ▌                                 0.5%

  → Data Skewness: 99% das queries caem na mesma partição
  → Sem ganho de performance — pior: 1 executor sobrecarregado

  Coluna: Year (para dados de voos 2007-2008)

  Partição 2007: ████████████████  ~7M voos
  Partição 2008: ████████████████  ~7M voos

  → Distribuição balanceada → boa candidata a partição ✓
```

> **Regra prática:** datas (Year, YearMonth) costumam ser boas escolhas porque  
> a distribuição tende a ser uniforme e as queries frequentemente filtram por período.

---

### Partição com múltiplas colunas

Quando uma única coluna não é suficiente, é possível particionar por duas colunas.  
Isso cria uma hierarquia de subdiretórios — com seus próprios desafios:

```
  PARTITIONED BY (Year, Origin)

  flight_data_year_origin/
  ├── Year=2007/
  │     ├── Origin=ATL/  part-001.parquet  (grande)
  │     ├── Origin=ORD/  part-002.parquet  (grande)
  │     ├── Origin=ACK/  part-003.parquet  (minúsculo)
  │     └── Origin=XYZ/  (vazio)
  └── Year=2008/
        └── ...

  → Partições populares: OK
  → Partições pequenas ou vazias: small files problem por partição
  → Solução: Range Partitioning agrupa valores em faixas
              ex: Origin A-F → pasta 1 | G-M → pasta 2 | N-Z → pasta 3
```

---

## 3. Setup

In [ ]:
%python
# Desativa o cache de IO para garantir que as queries meçam o tempo real de leitura
# e não resultados em cache — comparação de performance justa
spark.conf.set("spark.databricks.io.cache.enabled", False)

---
## 4. Explorando os dados

Usamos o dataset público de voos da ASA (Airlines) disponível no Databricks.  
Cada arquivo CSV contém ~7 milhões de registros de voos de um ano.

In [ ]:
-- Lista os arquivos disponíveis e seus tamanhos
-- Cada arquivo representa um ano de dados de voos
%fs
ls /databricks-datasets/asa/airlines

In [ ]:
%%sql
-- Exploração inicial: colunas disponíveis e amostra dos dados
SELECT * FROM CSV.`dbfs:/databricks-datasets/asa/airlines/2007.csv`
LIMIT 10;

> **Colunas relevantes para nossos testes:**  
> `Year` — baixa cardinalidade (2007, 2008) → candidata a partição  
> `Origin` — média/alta cardinalidade (centenas de aeroportos) → segunda dimensão de partição  
> `Distance` — coluna numérica usada nas queries de comparação (`avg`)

---

## 5. Tabela sem particionamento (baseline)

Criamos a tabela base sem nenhuma partição.  
O Delta Lake usará apenas o File Skipping via estatísticas do `_delta_log` para otimizar as queries.

In [ ]:
%%sql
DROP TABLE IF EXISTS flight_data;
CREATE TABLE flight_data (
    Year INT, Month INT, DayofMonth INT, DayofWeek INT, DepTime STRING, CRSDepTime INT, ArrTime STRING, CRSArrTime INT, UniqueCarrier STRING, FlightNum INT, TailNum STRING, ActualElapsedTime STRING, CRSElapsedTime STRING, AirTime STRING, ArrDelay STRING, DepDelay STRING, Origin STRING, Dest STRING, Distance INT, TaxiIn STRING, TaxiOut STRING, Cancelled INT, CancellationCode STRING, Diverted INT, CarrierDelay STRING, WeatherDelay STRING, NASDelay STRING, SecurityDelay STRING, LateAircraftDelay STRING
);

In [ ]:
%python
# Ingere 2 anos de dados (2007 e 2008) em uma única tabela sem partição
spark.read.csv("dbfs:/databricks-datasets/asa/airlines/{2007,2008}.csv", header=True, inferSchema=True)\
.write.mode("append").saveAsTable("flight_data")

---
## 6. Tabela particionada por Year

In [ ]:
%%sql
DROP TABLE IF EXISTS flight_data_year;
CREATE TABLE flight_data_year (
    Year INT, Month INT, DayofMonth INT, DayofWeek INT, DepTime STRING, CRSDepTime INT, ArrTime STRING, CRSArrTime INT, UniqueCarrier STRING, FlightNum INT, TailNum STRING, ActualElapsedTime STRING, CRSElapsedTime STRING, AirTime STRING, ArrDelay STRING, DepDelay STRING, Origin STRING, Dest STRING, Distance INT, TaxiIn STRING, TaxiOut STRING, Cancelled INT, CancellationCode STRING, Diverted INT, CarrierDelay STRING, WeatherDelay STRING, NASDelay STRING, SecurityDelay STRING, LateAircraftDelay STRING
)
PARTITIONED BY (Year);

In [ ]:
%%sql
-- A seção '#Partition Information' mostra a coluna escolhida para particionamento
DESC EXTENDED flight_data_year;

In [ ]:
%python
spark.read.csv("dbfs:/databricks-datasets/asa/airlines/{2007,2008}.csv", header=True, inferSchema=True)\
.write.mode("append").saveAsTable("flight_data_year")

In [ ]:
-- Estrutura de diretórios particionada por Year:
-- Year=2007/ e Year=2008/ como subpastas separadas
%fs
ls dbfs:/user/hive/warehouse/flight_data_year

---
### 6.1 Comparação de performance — sem partição vs particionado por Year

In [ ]:
%%sql
-- Baseline: sem partição
-- ⏱ Anote o scan time e number of files read na Spark UI
SELECT avg(Distance)
FROM flight_data
WHERE year = 2007;

In [ ]:
%%sql
-- Com particionamento por Year
-- ⏱ Anote o scan time e number of files read na Spark UI
SELECT avg(Distance)
FROM flight_data_year
WHERE year = 2007;

> **Resultado na Spark UI — um resultado surpreendente:**
>
> | Tabela | Arquivos lidos | Motivo |
> |---|---|---|
> | `flight_data` (sem partição) | 6 | File Skipping via stats do `_delta_log` |
> | `flight_data_year` (partição por Year) | 6 | Lê apenas a pasta `Year=2007/` |
>
> As duas queries leram o mesmo número de arquivos!  
> Isso demonstra um ponto crítico:
>
> **O Delta Log com File Skipping já é tão eficiente quanto o particionamento para tabelas menores.**  
> O `_delta_log` armazena `minValues` e `maxValues` de `Year` por arquivo,  
> eliminando automaticamente os arquivos de 2008 sem precisar de subdiretórios.
>
> ```
>   File Skipping (sem partição):      Partitioning (com partição):
>   _delta_log lê stats de Year        Spark abre apenas Year=2007/
>   min=2007, max=2007 → lê            ignora Year=2008/ completamente
>   min=2008, max=2008 → skip
>   Resultado: mesmo número de leituras
> ```
>
> 💡 **Conclusão:** particionamento não é bala de prata — o benefício só fica evidente  
> em tabelas muito grandes (> 1TB) onde o File Skipping sozinho não é suficiente.

---

## 7. Tabela particionada por Year e Origin

Adicionamos uma segunda dimensão de particionamento.  
Queries que filtram por `Year` **e** `Origin` agora leem apenas o subdiretório exato.

In [ ]:
%%sql
DROP TABLE IF EXISTS flight_data_year_origin;
CREATE TABLE flight_data_year_origin (
    Year INT, Month INT, DayofMonth INT, DayofWeek INT, DepTime STRING, CRSDepTime INT, ArrTime STRING, CRSArrTime INT, UniqueCarrier STRING, FlightNum INT, TailNum STRING, ActualElapsedTime STRING, CRSElapsedTime STRING, AirTime STRING, ArrDelay STRING, DepDelay STRING, Origin STRING, Dest STRING, Distance INT, TaxiIn STRING, TaxiOut STRING, Cancelled INT, CancellationCode STRING, Diverted INT, CarrierDelay STRING, WeatherDelay STRING, NASDelay STRING, SecurityDelay STRING, LateAircraftDelay STRING
)
PARTITIONED BY (Year, Origin);

In [ ]:
%python
# Ingestão mais demorada: o Spark precisa separar os dados em centenas de subpastas
# (2 anos × ~300 aeroportos de origem = ~600 subdiretórios)
spark.read.csv("dbfs:/databricks-datasets/asa/airlines/{2007,2008}.csv", header=True, inferSchema=True)\
.write.mode("append").saveAsTable("flight_data_year_origin")

In [ ]:
-- Hierarquia de subdiretórios: Year=2007/ contém um subdiretório por aeroporto de origem
%fs
ls dbfs:/user/hive/warehouse/flight_data_year_origin/Year=2007/

### 7.1 Comparação de performance — com filtro em dois campos

In [ ]:
%%sql
-- Tabela sem partição — File Skipping via _delta_log
-- ⏱ Spark UI: number of files read e scan time
SELECT avg(Distance)
FROM flight_data
WHERE year = 2007
AND Origin = 'ACK';

In [ ]:
%%sql
-- Tabela particionada por Year e Origin
-- Spark abre APENAS Year=2007/Origin=ACK/ — ignora todos os outros ~599 subdiretórios
-- ⏱ Spark UI: number of files read e scan time
SELECT avg(Distance)
FROM flight_data_year_origin
WHERE year = 2007
AND Origin = 'ACK';

> **Resultado na Spark UI:**
>
> | Tabela | Arquivos lidos | Impacto |
> |---|---|---|
> | `flight_data` (sem partição) | 6 | File Skipping via stats |
> | `flight_data_year_origin` | 2 | Apenas `Year=2007/Origin=ACK/` |
>
> Aqui o benefício é claro: **6 → 2 arquivos**. Em escala (petabytes, milhares de partições),  
> essa diferença se torna enorme. Porém, o custo também se revela:
>
> ```
>   ACK (Nantucket) é um aeroporto pequeno → poucos voos → arquivo minúsculo
>   ATL (Atlanta) é um hub → muitos voos  → arquivo grande
>   → Data Skewness: distribuição extremamente desigual entre partições
> ```

---

## 8. Analisando a distribuição dos dados

Antes de checar a distribuição, vamos confirmar o número de arquivos em cada tabela:

In [ ]:
%%sql
-- 12 arquivos
DESC DETAIL flight_data;

In [ ]:
%%sql
-- ~12 arquivos
DESC DETAIL flight_data_year;

In [ ]:
%%sql
-- ⚠️ ~3371 arquivos — o particionamento por Origin explodiu o número de arquivos
-- Cada aeroporto de origem × cada ano = centenas de Parquets pequenos
DESC DETAIL flight_data_year_origin;

> **O número 3371 ilustra exatamente o problema:**  
> Particionamento por uma coluna de média-alta cardinalidade (`Origin`) gerou  
> um small files problem em nível de partição — cada pasta com pouquíssimos dados.

---

### 8.1 Análise detalhada da distribuição por pasta

In [ ]:
%python
# Análise de distribuição: num_files e total_size por pasta Year/Origin
# Mostra o desequilíbrio entre aeroportos grandes (hubs) e pequenos
from pyspark.sql import functions as F

root_path = "dbfs:/user/hive/warehouse/flight_data_year_origin/"
year_origin_folders = dbutils.fs.ls(root_path)

file_info = []

for year_folder in year_origin_folders:
    if year_folder.isDir() and "_delta_log" not in year_folder.name:
        origin_folders = dbutils.fs.ls(year_folder.path)
        for origin_folder in origin_folders:
            if origin_folder.isDir() and "_delta_log" not in origin_folder.name:
                files_in_folder = dbutils.fs.ls(origin_folder.path)
                total_size = sum(file.size for file in files_in_folder)
                num_files = len(files_in_folder)

                year = year_folder.name.split("=")[1]
                origin = origin_folder.name.split("=")[1]

                file_info.append((origin_folder.path, num_files, total_size/(1024*1024), year, origin))

df = spark.createDataFrame(file_info, ["path_to_folder", "num_files", "total_size", "year", "origin"])
df.display()

> **O que observar no resultado:**
> - Aeroportos hub (ATL, ORD, LAX) → `total_size` alto, `num_files` maior
> - Aeroportos pequenos (ACK, SCC, CDV) → `total_size` próximo de zero, `num_files` = 1
> - Essa assimetria é o **data skewness** — a maioria das partições é irrelevante
>   e acumula arquivos minúsculos sem nenhum benefício de leitura

---

## 9. Key Points — Partitioning

```
┌──────────────────────────────────────────────────────────────────┐
│              QUANDO O PARTICIONAMENTO AJUDA                      │
│                                                                  │
│  ✅ Tabela > 1 TB de dados                                       │
│  ✅ Cada partição terá pelo menos 1 GB de dados                  │
│  ✅ Coluna com baixa-média cardinalidade                         │
│  ✅ Distribuição equilibrada entre os valores da coluna          │
│  ✅ Queries quase sempre filtram pela coluna de partição         │
│                                                                  │
│              ARMADILHAS DO PARTICIONAMENTO                       │
│                                                                  │
│  ⚠️  Tabela < 1 TB: File Skipping do Delta Log já é suficiente   │
│  ⚠️  Data Skewness: partições com distribuição desigual          │
│  ⚠️  Small Files por partição (ex: 3371 arquivos no exemplo)     │
│  ⚠️  Requer reescrita total da tabela para mudar a partição      │
│  ⚠️  Se o padrão de queries mudar, a partição pode prejudicar    │
└──────────────────────────────────────────────────────────────────┘
```

### Comparação das estratégias de otimização

| Estratégia | Cardinalidade | Tamanho mínimo | Modifica estrutura física | Requer reescrita |
|---|---|---|---|---|
| **File Skipping** (automático) | Qualquer | Qualquer | ❌ | ❌ |
| **OPTIMIZE / Auto Compact** | N/A | Qualquer | Sim (compacta) | ❌ |
| **Partitioning** | Baixa / Média | > 1 TB total / > 1 GB por partição | Sim (subdiretórios) | ✅ |
| **Z-Ordering** | Alta | Qualquer | Sim (co-localiza) | ❌ |